# Data Science Final Project – Ford GoBike Trip Analysis

**Student:** Ahmed Antar  
**Module:** Data Science  
**Year:** 2026

## Introduction
I am using the February 2019 Ford GoBike trip data in this project. The main idea is to understand when bikes are used, how long trips take, and whether usage changes between different groups of riders.

## Questions for the analysis
1. What is the structure and quality of the dataset?
2. Which days and hours have the most trips?
3. What does the trip-duration distribution look like?
4. How does duration differ between Subscribers and Customers?
5. Is the usage pattern different on weekdays and weekends?
6. How are gender and user type related in the recorded trips?
7. Which start stations are most common?
8. Is there a clear relationship between age and duration?
9. What patterns appear when time, user type and duration are combined?

## 1. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 2. Load the dataset

Keep `fordgobike-tripdataFor201902.csv` in the same folder as this notebook. A few common paths are checked below so I can move the project between computers.

In [ ]:
possible_paths = [
    "fordgobike-tripdataFor201902.csv",
    "./data/fordgobike-tripdataFor201902.csv",
    "/mnt/data/fordgobike-tripdataFor201902.csv",
    "C:/Games/fordgobike-tripdataFor201902.csv"
]

data_path = next((p for p in possible_paths if os.path.exists(p)), None)

if data_path is None:
    raise FileNotFoundError(
        "CSV not found. Put fordgoBike-tripdataFor201902.csv beside the notebook "
        "or change possible_paths."
    )

df = pd.read_csv(data_path)
print("File:", data_path)
print("Shape:", df.shape)

## 3. First look at the data

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

df.head(10)

In [ ]:
df.describe(include="all").T

In [ ]:
quality = pd.DataFrame({
    "type": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "unique": df.nunique()
}).sort_values("missing", ascending=False)
quality

### First observations
The first checks show the number of records, column types and missing values. Before making charts I will clean the basic data issues and create features that are easier to interpret.

## 4. Cleaning

In [ ]:
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

print("Exact duplicates:", df.duplicated().sum())
df = df.drop_duplicates().copy()
print("Duplicates after cleaning:", df.duplicated().sum())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]

For categorical variables I use the mode for missing values. For birth year I use the median. For station names I use `Unknown` instead of creating a station that was not in the source data.

In [ ]:
for col in ["user_type", "member_gender", "bike_share_for_all_trip"]:
    df[col] = df[col].fillna(df[col].mode()[0])

df["member_birth_year"] = df["member_birth_year"].fillna(
    df["member_birth_year"].median()
)

for col in ["start_station_name", "end_station_name"]:
    df[col] = df[col].fillna("Unknown")

for col in ["start_station_id", "end_station_id"]:
    df[col] = df[col].fillna(0)

for col in ["user_type", "member_gender", "bike_share_for_all_trip"]:
    df[col] = df[col].astype("category")

print("Remaining missing values:", int(df.isna().sum().sum()))

## 5. Feature engineering

In [ ]:
df["duration_min"] = df["duration_sec"] / 60
df["start_hour"] = df["start_time"].dt.hour
df["day_name"] = df["start_time"].dt.day_name()

day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
df["day_name"] = pd.Categorical(df["day_name"], categories=day_order, ordered=True)

df["is_weekend"] = df["start_time"].dt.dayofweek >= 5

# The trip is from 2019, so age should be based on the trip year.
df["user_age"] = df["start_time"].dt.year - df["member_birth_year"]

df[["duration_min","start_hour","day_name","is_weekend","user_age"]].head()

In [ ]:
print(df[["duration_min", "user_age"]].describe())
print("Age below 16:", (df["user_age"] < 16).sum())
print("Age above 80:", (df["user_age"] > 80).sum())

Some birth-year entries create unrealistic ages. For age-based charts I use a separate dataframe with ages from 16 to 80 and positive trip duration. This leaves the original cleaned dataframe available.

In [ ]:
analysis_df = df[
    df["user_age"].between(16, 80) &
    df["duration_min"].gt(0)
].copy()

print("Original cleaned rows:", len(df))
print("Rows used for analysis:", len(analysis_df))

# Univariate Exploration

## 6. Trips by user type

In [ ]:
counts = analysis_df["user_type"].value_counts()

plt.figure(figsize=(7,5))
sns.countplot(data=analysis_df, x="user_type", order=counts.index)
plt.title("Trips by User Type")
plt.xlabel("User type")
plt.ylabel("Number of trips")
plt.show()

print(counts)

## 7. Trips by day

In [ ]:
day_counts = analysis_df["day_name"].value_counts().reindex(day_order)

plt.figure(figsize=(10,5))
sns.barplot(x=day_counts.index, y=day_counts.values)
plt.title("Trips by Day of the Week")
plt.xlabel("Day")
plt.ylabel("Number of trips")
plt.xticks(rotation=25)
plt.show()

day_counts

## 8. Trips by starting hour

In [ ]:
hour_counts = analysis_df["start_hour"].value_counts().sort_index()

plt.figure(figsize=(11,5))
sns.barplot(x=hour_counts.index, y=hour_counts.values)
plt.title("Trips by Starting Hour")
plt.xlabel("Hour")
plt.ylabel("Number of trips")
plt.xticks(range(24))
plt.show()

print("Busiest starting hour:", hour_counts.idxmax())

## 9. Trip duration

In [ ]:
print(analysis_df["duration_min"].describe())

plt.figure(figsize=(10,5))
sns.histplot(data=analysis_df, x="duration_min", bins=50)
plt.xlim(0, analysis_df["duration_min"].quantile(.99))
plt.title("Trip Duration Distribution")
plt.xlabel("Duration (minutes)")
plt.ylabel("Number of trips")
plt.show()

Most trips are short, while a smaller number of rides last much longer. This is why the distribution is not perfectly symmetric.

## 10. Rider age

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=analysis_df, x="user_age", bins=30)
plt.title("Rider Age Distribution")
plt.xlabel("Age")
plt.ylabel("Number of trips")
plt.show()

print("Mean age:", round(analysis_df["user_age"].mean(), 1))
print("Median age:", round(analysis_df["user_age"].median(), 1))

## 11. Member gender

In [ ]:
gender_counts = analysis_df["member_gender"].value_counts()

plt.figure(figsize=(8,5))
sns.countplot(data=analysis_df, x="member_gender", order=gender_counts.index)
plt.title("Trips by Member Gender")
plt.xlabel("Gender")
plt.ylabel("Number of trips")
plt.show()

gender_counts

# Bivariate Exploration

## 12. Duration by user type

In [ ]:
avg_duration = (
    analysis_df.groupby("user_type", observed=True)["duration_min"]
    .mean()
    .sort_values(ascending=False)
)

print(avg_duration)

plt.figure(figsize=(7,5))
sns.barplot(x=avg_duration.index, y=avg_duration.values)
plt.title("Average Trip Duration by User Type")
plt.xlabel("User type")
plt.ylabel("Average duration (minutes)")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=analysis_df, x="user_type", y="duration_min")
plt.ylim(0, analysis_df["duration_min"].quantile(.98))
plt.title("Trip Duration Distribution by User Type")
plt.xlabel("User type")
plt.ylabel("Duration (minutes)")
plt.show()

The boxplot is useful because the mean can be moved by unusually long rides.

## 13. User type across the week

In [ ]:
user_day = pd.crosstab(
    analysis_df["day_name"], analysis_df["user_type"]
).reindex(day_order)

user_day.plot(kind="bar", figsize=(11,5))
plt.title("Trips by Day and User Type")
plt.xlabel("Day")
plt.ylabel("Number of trips")
plt.xticks(rotation=25)
plt.legend(title="User type")
plt.show()

## 14. Hourly pattern for each user type

In [ ]:
hour_user = pd.crosstab(
    analysis_df["start_hour"], analysis_df["user_type"]
).sort_index()

plt.figure(figsize=(11,6))
for group in hour_user.columns:
    plt.plot(hour_user.index, hour_user[group], marker="o", label=str(group))

plt.title("Hourly Trips by User Type")
plt.xlabel("Starting hour")
plt.ylabel("Number of trips")
plt.xticks(range(24))
plt.legend(title="User type")
plt.show()

## 15. Gender and user type

In [ ]:
gender_user = pd.crosstab(
    analysis_df["member_gender"], analysis_df["user_type"]
)

plt.figure(figsize=(8,5))
sns.heatmap(gender_user, annot=True, fmt="d")
plt.title("Gender by User Type")
plt.xlabel("User type")
plt.ylabel("Gender")
plt.show()

gender_user

## 16. Age vs duration

In [ ]:
sample_df = analysis_df.sample(
    min(15000, len(analysis_df)), random_state=42
)

plt.figure(figsize=(10,6))
sns.scatterplot(
    data=sample_df,
    x="user_age",
    y="duration_min",
    alpha=.25
)
plt.ylim(0, analysis_df["duration_min"].quantile(.98))
plt.title("Rider Age vs Trip Duration")
plt.xlabel("Age")
plt.ylabel("Duration (minutes)")
plt.show()

print(
    "Correlation:",
    round(analysis_df[["user_age","duration_min"]].corr().iloc[0,1], 3)
)

## 17. Most common starting stations

In [ ]:
top_stations = (
    analysis_df["start_station_name"]
    .value_counts()
    .head(10)
    .sort_values()
)

plt.figure(figsize=(10,6))
sns.barplot(x=top_stations.values, y=top_stations.index)
plt.title("Top 10 Starting Stations")
plt.xlabel("Number of trips")
plt.ylabel("Station")
plt.show()

top_stations.sort_values(ascending=False)

# Multivariate Exploration

## 18. Day and hour heatmap

In [ ]:
day_hour = pd.crosstab(
    analysis_df["day_name"], analysis_df["start_hour"]
).reindex(day_order)

plt.figure(figsize=(14,6))
sns.heatmap(day_hour)
plt.title("Trip Count by Day and Starting Hour")
plt.xlabel("Hour")
plt.ylabel("Day")
plt.show()

## 19. Duration by user type and weekend

In [ ]:
weekend_duration = (
    analysis_df.groupby(
        ["user_type","is_weekend"], observed=True
    )["duration_min"].mean().reset_index()
)

weekend_duration["period"] = weekend_duration["is_weekend"].map({
    False: "Weekday", True: "Weekend"
})

plt.figure(figsize=(9,5))
sns.barplot(
    data=weekend_duration,
    x="user_type",
    y="duration_min",
    hue="period"
)
plt.title("Average Duration: Weekday vs Weekend")
plt.xlabel("User type")
plt.ylabel("Average duration (minutes)")
plt.show()

weekend_duration

## 20. Age group, user type and duration

In [ ]:
age_groups = analysis_df.copy()
age_groups["age_group"] = pd.cut(
    age_groups["user_age"],
    bins=[15,25,35,45,55,65,80],
    labels=["16-25","26-35","36-45","46-55","56-65","66-80"]
)

age_summary = (
    age_groups.groupby(
        ["age_group","user_type"], observed=True
    )["duration_min"].mean().reset_index()
)

plt.figure(figsize=(11,5))
sns.lineplot(
    data=age_summary,
    x="age_group",
    y="duration_min",
    hue="user_type",
    marker="o"
)
plt.title("Average Duration by Age Group and User Type")
plt.xlabel("Age group")
plt.ylabel("Average duration (minutes)")
plt.show()

age_summary

## 21. Selected numeric correlations

In [ ]:
numeric_cols = [
    "duration_sec", "duration_min", "user_age", "start_hour",
    "start_station_latitude", "start_station_longitude",
    "end_station_latitude", "end_station_longitude"
]

corr = analysis_df[numeric_cols].corr()

plt.figure(figsize=(10,7))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Correlation Matrix")
plt.show()

corr.round(2)

# 22. Main findings

- Trip duration is concentrated on shorter rides, with some long rides creating a right-skewed distribution.
- The number of trips changes a lot by starting hour and by day of the week.
- Subscribers and Customers can have different duration distributions.
- Weekend and weekday behavior can be compared using both trip counts and average duration.
- Gender and user type show how the recorded trips are distributed across these categories.
- A few starting stations account for a large amount of the recorded activity.
- Age and duration can be compared statistically, but correlation by itself does not establish causation.

# 23. Limitations

- The file represents February 2019, so it is one month rather than a full-year dataset.
- Trip counts are not the same as unique customer counts.
- Filled values are estimates where the source had missing data.
- Station and bike IDs are identifiers and are not treated as meaningful continuous measurements.
- The analysis is descriptive and does not explain the causes behind the patterns.

# 24. Conclusion

I cleaned the source data, handled missing values and duplicates, converted timestamps, and created features for duration, starting hour, day, weekend status and age. After that I used univariate, bivariate and multivariate visualizations to look at time patterns, user groups, duration, age and gender.

The project shows a complete basic workflow for a data-science analysis: define questions, inspect the data, clean it, create useful features, visualize relationships and explain the findings.

## End of project

**Libraries used:** pandas, numpy, matplotlib, seaborn